# Missing values

IMP-03. The Instructions sheet asked collectors not to leave blanks, so besides empty cells this also checks text in number columns, copied beta-HCG values, impossible zeros, the 1.99 values and highlighted cells. Nothing in the data is changed here.

In [1]:
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"
pd.set_option("display.max_rows", 60)

df = pd.read_excel(DATA_FILE, sheet_name="Full_new")
df.columns = df.columns.str.strip()          # remove hidden leading/trailing spaces found in IMP-02

LABEL = "PCOS (Y/N)"
ID_COLS = ["Sl. No", "Patient File No."]
STRAY = [c for c in df.columns if c.startswith("Unnamed")]
FEATURES = [c for c in df.columns if c not in ID_COLS + [LABEL] + STRAY]

print(f"Records: {len(df)} | features checked: {len(FEATURES)} | excluded stray column(s): {STRAY}")

Records: 541 | features checked: 41 | excluded stray column(s): ['Unnamed: 44']


## Blank cells

In [2]:
blanks = df[FEATURES].isna().sum()
blanks_by_class = df.groupby(LABEL)[FEATURES].apply(lambda g: g.isna().sum()).T

type1 = pd.DataFrame({
    "blank_cells": blanks,
    "percent": (100 * blanks / len(df)).round(2),
    "in_not_pcos": blanks_by_class[0],
    "in_pcos": blanks_by_class[1],
})
type1[type1["blank_cells"] > 0]

,blank_cells,percent,in_not_pcos,in_pcos
Marraige Status (Yrs),1,0.18,0,1
Fast food (Y/N),1,0.18,1,0


## Text in number columns

`pd.to_numeric(errors="coerce")` turns non-numbers into NaN. If a value becomes a number once a trailing full stop is removed, it's a typo rather than a missing value.

In [3]:
text_cells = []
for col in FEATURES:
    if df[col].dtype == "object":
        as_number = pd.to_numeric(df[col], errors="coerce")
        became_blank = as_number.isna() & df[col].notna()
        for idx in df.index[became_blank]:
            raw = str(df.at[idx, col])
            recoverable = pd.notna(pd.to_numeric(raw.rstrip("."), errors="coerce"))
            text_cells.append({
                "patient_file_no": df.at[idx, "Patient File No."],
                "column": col,
                "value_in_file": repr(raw),
                "type": "typo (recoverable)" if recoverable else "unknown (missing)",
                "pcos": df.at[idx, LABEL],
            })
type2 = pd.DataFrame(text_cells)
type2

,patient_file_no,column,value_in_file,type,pcos
0,124,II beta-HCG(mIU/mL),'1.99.',typo (recoverable),1
1,306,AMH(ng/mL),'a',unknown (missing),0


## beta-HCG II copied from beta-HCG I

Instructions rule 10 says to repeat Case I when only one test exists.

In [4]:
hcg1 = df["I   beta-HCG(mIU/mL)"]
hcg2 = pd.to_numeric(df["II    beta-HCG(mIU/mL)"], errors="coerce")
identical = hcg1 == hcg2

type3 = pd.DataFrame({
    "records": [identical.sum(), (identical & (hcg1 == 1.99)).sum(), (identical & (hcg1 != 1.99)).sum()],
}, index=["II identical to I", "  of which both 1.99", "  of which another value"])
display(type3)
print("Identical, by class:", df[identical].groupby(LABEL).size().to_dict())

,records
II identical to I,311
of which both 1.99,179
of which another value,132


Identical, by class: {0: 213, 1: 98}


## Impossible zeros

Only columns where a real measurement can't be zero (labs, body measurements, vital signs, endometrium).

In [5]:
ZERO_IMPOSSIBLE = [
    "Weight (Kg)", "Height(Cm)", "BMI", "Pulse rate(bpm)", "RR (breaths/min)", "Hb(g/dl)",
    "I   beta-HCG(mIU/mL)", "FSH(mIU/mL)", "LH(mIU/mL)", "Hip(inch)", "Waist(inch)",
    "TSH (mIU/L)", "AMH(ng/mL)", "PRL(ng/mL)", "Vit D3 (ng/mL)", "PRG(ng/mL)", "RBS(mg/dl)",
    "BP _Systolic (mmHg)", "BP _Diastolic (mmHg)", "Endometrium (mm)",
]
zero_cells = []
for col in ZERO_IMPOSSIBLE:
    values = pd.to_numeric(df[col], errors="coerce")
    for idx in df.index[values == 0]:
        zero_cells.append({"patient_file_no": df.at[idx, "Patient File No."], "column": col,
                           "pcos": df.at[idx, LABEL]})
type4 = pd.DataFrame(zero_cells)
type4

,patient_file_no,column,pcos
0,217,Vit D3 (ng/mL),1
1,40,Endometrium (mm),0
2,83,Endometrium (mm),0


Check that average follicle size is only 0 when the follicle count is also 0.

In [6]:
for side in ["L", "R"]:
    count = df[f"Follicle No. ({side})"]
    size = df[f"Avg. F size ({side}) (mm)"]
    print(f"{side}: size = 0 in {(size == 0).sum()} records; of those, count = 0 in {((size == 0) & (count == 0)).sum()}")

L: size = 0 in 7 records; of those, count = 0 in 7
R: size = 0 in 1 records; of those, count = 0 in 1


## The value 1.99

Probably the lab detection limit (<2), i.e. a real low value.

In [7]:
numeric = df[FEATURES].apply(pd.to_numeric, errors="coerce")
at_199 = (numeric == 1.99).sum()
at_199[at_199 > 0].to_frame("cells equal to 1.99")

,cells equal to 1.99
I beta-HCG(mIU/mL),191
II beta-HCG(mIU/mL),307
FSH(mIU/mL),2
LH(mIU/mL),4
TSH (mIU/L),1
AMH(ng/mL),1


## Highlighted cells

pandas doesn't read cell colours, so openpyxl is used here. White and the green PCOS marker are ignored.

In [8]:
wb = load_workbook(DATA_FILE)
ws = wb["Full_new"]
header = [str(c.value).strip() if c.value is not None else None for c in ws[1]]

IGNORE = {"FFFFFFFF", "FF00FF00"}      # white, and the green PCOS marker
highlighted = []
for row in ws.iter_rows(min_row=2):
    for cell in row:
        fill = cell.fill
        if fill.fill_type and fill.fgColor.type == "rgb" and fill.fgColor.rgb not in IGNORE:
            idx = cell.row - 2                                   # Excel row 2 = first data row
            highlighted.append({"excel_row": cell.row,
                                "patient_file_no": df.at[idx, "Patient File No."],
                                "column": header[cell.column - 1],
                                "colour": fill.fgColor.rgb,
                                "value": cell.value})
type6 = pd.DataFrame(highlighted)

by_colour = type6.groupby("colour").agg(
    cells=("column", "size"),
    records=("patient_file_no", lambda s: sorted(int(x) for x in s.unique())),
    columns=("column", "nunique"),
)
display(by_colour)

legend = {c.fill.fgColor.rgb: wb["Instructions"].cell(row=c.row, column=2).value
          for row in wb["Instructions"].iter_rows() for c in row
          if c.fill.fill_type and c.fill.fgColor.type == "rgb" and c.fill.fgColor.rgb not in {"FFFFFFFF", "00000000"}}
print("Legend colours in the Instructions sheet:")
for colour, rule in legend.items():
    print(f"  {colour}: {rule.strip()}")

,cells,records,columns
colour,,,
FFDD7E6B,22,"[307, 308, 309, 310, 311, 312, 313, 314, 315, ...",1
FFE6B8AF,66,"[442, 443]",32
FFEFEFEF,44,[3],43


Legend colours in the Instructions sheet:
  FFFFE599: Manipulated datas if any need to be highlighted using the following colour
  FFFF0000: In case sparing a blank space kindly fill the cell with red colour
  FF00FF00: If the patient is PCOS Fill the cell with green colour for easier reference


## Missing values per feature

confirmed = blank cells + non-numeric text; possible = impossible zeros + copied beta-HCG II. Highlighted cells are shown for reference only.

In [9]:
summary = pd.DataFrame(index=FEATURES)
summary["blank"] = type1["blank_cells"]
unknown_text = type2[type2["type"] == "unknown (missing)"] if len(type2) else type2
summary["text_unknown"] = unknown_text.groupby("column").size() if len(unknown_text) else 0
summary["confirmed_missing"] = summary["blank"] + summary["text_unknown"].fillna(0)
summary["impossible_zero"] = type4.groupby("column").size() if len(type4) else 0
summary.loc["II    beta-HCG(mIU/mL)", "copied_from_I"] = identical.sum()
summary["highlighted"] = type6.groupby("column").size() if len(type6) else 0
summary = summary.fillna(0).astype(int)
summary["percent_confirmed"] = (100 * summary["confirmed_missing"] / len(df)).round(2)

# highlighted is shown for information only; it does not by itself mark a feature as missing
flagged = summary[(summary[["confirmed_missing", "impossible_zero", "copied_from_I"]] > 0).any(axis=1)]
display(flagged)
print(f"Features with confirmed or possible missing values: {len(flagged)} of {len(summary)}")
print(f"Confirmed missing cells in total: {summary['confirmed_missing'].sum()} "
      f"({100 * summary['confirmed_missing'].sum() / (len(df) * len(FEATURES)):.3f}% of all feature cells)")

,blank,text_unknown,confirmed_missing,impossible_zero,copied_from_I,highlighted,percent_confirmed
Marraige Status (Yrs),1,0,1,0,0,1,0.18
II beta-HCG(mIU/mL),0,0,0,0,311,3,0.00
AMH(ng/mL),0,1,1,0,0,3,0.18
Vit D3 (ng/mL),0,0,0,1,0,25,0.00
Fast food (Y/N),1,0,1,0,0,3,0.18
Endometrium (mm),0,0,0,2,0,3,0.00


Features with confirmed or possible missing values: 6 of 41
Confirmed missing cells in total: 3 (0.014% of all feature cells)


## Observations

- Only 3 values are really missing: one in marriage years, one in fast food, and one in AMH (written as 'a'). That's 0.014% of the data, so almost nothing is missing.
- Because of this, the dataset can't show on its own how my models cope with missing information. For the missing-information experiments I'll have to hide data on purpose.
- '1.99.' in beta-HCG II is just a typo for 1.99, not a missing value.
- In 311 records (57%), beta-HCG II is exactly the same as beta-HCG I. The instructions say to copy the first value if only one test was done, so the second test probably wasn't done in many of these.
- Vitamin D3 is 0 for patient 217, and endometrium is 0 for patients 40 and 83. These can't really be 0, so they're probably missing values.
- Average follicle size is only 0 when the follicle count is also 0, so those zeros make sense and aren't missing.
- The value 1.99 comes up a lot, mostly in beta-HCG. I think it's the lowest value the lab can measure, so I'm treating it as a real low result.
- Some cells are highlighted in a reddish colour: 22 vitamin D3 values (patients 307-328) and two whole records (442 and 443). The colour is close to the "blank, filled in" red in the instructions but not the same, so I can't be sure what it means. I'll mention it as a limitation.
- Overall, 6 of the 41 features have missing or possibly missing values.